In [ ]:
import torch
if '2.7' not in torch.__version__:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q',
        'torch==2.7.0','torchvision','torchaudio',
        '--index-url','https://download.pytorch.org/whl/cu128'],capture_output=True)
    import IPython; IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print(f'PyTorch {torch.__version__} — sm_120 ready.')


In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed: {seed}')
set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')


In [ ]:
class Config:
    # ── Paths ──────────────────────────────────────────────────────
    data_path    = "/kaggle/input/datasets/piyush1718s/pems08/PEMS08.npz"
    adj_csv_path = "/kaggle/input/datasets/piyush1718s/pems08csv/PEMS08.csv"
    best_path    = 'gwnet_v25_best.pt'

    # ── Dataset ────────────────────────────────────────────────────
    num_nodes   = 170
    in_features = 3
    seq_len     = 18       # 90 min input history
    pred_len    = 12       # 60 min prediction horizon
    feature_idx = 0
    train_ratio = 0.7
    val_ratio   = 0.1
    noise_std   = 0.0
    # Multi-period offsets (5-min slots)
    HOUR_OFFSET = 12       # 1 hour ago
    DAY_OFFSET  = 288      # 24 hours ago

    # ── Model (proven from v20) ─────────────────────────────────────
    d_model    = 96
    d_skip     = 512
    d_stf      = 256       # STFormer internal dim (projected from d_skip)
    d_end      = 512
    d_time     = 128       # tod + dow embedding dim each
    n_layers   = 16        # WaveBlocks: dilations [1,2,4,8] x 4
    kernel_size = 4
    adp_emb    = 20
    gcn_order  = 4
    n_supports = 3
    dropout    = 0.15
    # STFormer
    stf_layers = 2         # L=2 spatial+temporal transformer layers
    stf_heads  = 4         # attention heads

    # ── Training ───────────────────────────────────────────────────
    batch_size   = 64
    lr           = 7e-4    # slightly below v20 for stability with STFormer
    warmup_eps   = 5
    epochs       = 200
    patience     = 40
    weight_decay = 1e-4
    huber_delta  = 1.0     # Huber threshold (normalised space)

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GWNet v25 | d={cfg.d_model} | d_skip={cfg.d_skip} | d_stf={cfg.d_stf}')
print(f'  n_layers={cfg.n_layers} | STFormer L={cfg.stf_layers} | {device}')
print(f'  3-stream input: recent + 1h-ago + 24h-ago | seq={cfg.seq_len}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA — 3-stream dataset
# ═══════════════════════════════════════════════════════════════════════════

def load_pems08(cfg):
    import pandas as pd, os
    raw  = np.load(cfg.data_path)
    data = raw['data'].astype(np.float32)
    T, N, F = data.shape
    print(f'Shape: {data.shape}')
    mean_np = data.mean(axis=0)                     # (N, F)
    std_np  = data.std(axis=0)  + 1e-8
    data_norm = (data - mean_np[None]) / std_np[None]

    # ── Adjacency ──────────────────────────────────────────────────
    A_dist = None
    if os.path.exists(cfg.adj_csv_path):
        df = pd.read_csv(cfg.adj_csv_path)
        A_raw = np.zeros((N,N), dtype=np.float32)
        for _, r in df.iterrows():
            i,j,c = int(r['from']), int(r['to']), float(r['cost'])
            if i<N and j<N:
                A_raw[i,j]=c; A_raw[j,i]=c
        nz    = A_raw[A_raw>0]
        sigma = nz.std() if len(nz)>0 else 1.0
        A_dist = np.exp(-(A_raw**2)/(sigma**2+1e-8))
        np.fill_diagonal(A_dist, 0.0)
        A_dist = A_dist / (A_dist.sum(1,keepdims=True)+1e-8)
        print(f'Adj: {(A_dist>0).sum()} non-zero entries, avg_deg={(A_dist>0).sum()/N:.1f}')
    else:
        A_dist = np.eye(N, dtype=np.float32)
        print('WARNING: CSV not found, using identity adj')
    return data_norm, mean_np, std_np, A_dist


class TrafficDataset3S(Dataset):
    """
    3-stream: (x_rec, x_hour, x_day, y, tod, dow)
      x_rec : data[i : i+S]             — most recent S steps
      x_hour: data[i-12 : i-12+S]       — same window 1h ago
      x_day : data[i-288 : i-288+S]     — same window 24h ago
    Requires i >= 288 so we always have valid 24h lookback.
    """
    def __init__(self, data, seq_len, pred_len, feature_idx,
                 hour_off, day_off, split_start=0, split_end=None, training=False):
        self.data, self.sl, self.pl = data, seq_len, pred_len
        self.fi, self.ho, self.do   = feature_idx, hour_off, day_off
        self.training               = training
        T = len(data)
        split_end = split_end or T
        eff_start = max(split_start, day_off)   # need 24h lookback
        self.idx  = list(range(eff_start, split_end - seq_len - pred_len + 1))

    def __len__(self): return len(self.idx)

    def __getitem__(self, n):
        i, S = self.idx[n], self.sl
        x_rec  = self.data[i        : i+S             ].copy()
        x_hour = self.data[i-self.ho: i-self.ho+S     ].copy()
        x_day  = self.data[i-self.do: i-self.do+S     ].copy()
        y      = self.data[i+S : i+S+self.pl, :, self.fi].copy()
        tod    = np.array([(i+t)%288       for t in range(S)], dtype=np.int64)
        dow    = np.array([((i+t)//288)%7  for t in range(S)], dtype=np.int64)
        return (torch.from_numpy(x_rec),  torch.from_numpy(x_hour),
                torch.from_numpy(x_day),  torch.from_numpy(y),
                torch.from_numpy(tod),    torch.from_numpy(dow))


def build_dataloaders(cfg):
    set_seed()
    data, mean_np, std_np, A_dist = load_pems08(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    ds_kw = dict(data=data, seq_len=cfg.seq_len, pred_len=cfg.pred_len,
                 feature_idx=cfg.feature_idx,
                 hour_off=cfg.HOUR_OFFSET, day_off=cfg.DAY_OFFSET)
    dl_tr = DataLoader(TrafficDataset3S(**ds_kw, split_start=0,  split_end=t1, training=True),  shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset3S(**ds_kw, split_start=t1, split_end=t2, training=False), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset3S(**ds_kw, split_start=t2, split_end=T,  training=False), shuffle=False, **kw)
    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_dist

print('3-stream dataset ready.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BUILDING BLOCKS
# ═══════════════════════════════════════════════════════════════════════════

class DiffusionGCN(nn.Module):
    """K-hop diffusion GCN (same as v20, proven working)."""
    def __init__(self, d_in, d_out, n_supports=3, order=2, dropout=0.15):
        super().__init__()
        self.mlp   = nn.Linear(d_in * (1 + n_supports * order), d_out)
        self.drop  = nn.Dropout(dropout)
        self.order = order
    def forward(self, x, supports):   # x: (B*S, N, d)
        hs = [x]
        for A in supports:
            h = x
            for _ in range(self.order):
                h = torch.einsum('nm,bmd->bnd', A, h)
                hs.append(h)
        return self.drop(self.mlp(torch.cat(hs, -1)))


class WaveBlock(nn.Module):
    """Gated TCN + DiffusionGCN + BN + skip/residual (same as v20)."""
    def __init__(self, d_model, d_skip, kernel_size, dilation,
                 n_supports, gcn_order, dropout):
        super().__init__()
        self.ks, self.dil = kernel_size, dilation
        kw = dict(kernel_size=(1,kernel_size), dilation=(1,dilation))
        self.f_conv   = nn.Conv2d(d_model, d_model, **kw)
        self.g_conv   = nn.Conv2d(d_model, d_model, **kw)
        self.gcn      = DiffusionGCN(d_model, d_model, n_supports, gcn_order, dropout)
        self.bn        = nn.BatchNorm2d(d_model)
        self.drop      = nn.Dropout(dropout)
        self.skip_conv = nn.Conv2d(d_model, d_skip,  (1,1))
        self.res_conv  = nn.Conv2d(d_model, d_model, (1,1))

    def forward(self, x, sup):         # x: (B, d, N, S)
        res   = x
        pad   = (self.ks - 1) * self.dil
        xp    = F.pad(x, [pad, 0])
        f = torch.tanh   (self.f_conv(xp))
        g = torch.sigmoid(self.g_conv(xp))
        x = self.drop(f * g)
        B, d, N, S = x.shape
        xg = x.permute(0,3,2,1).reshape(B*S, N, d)
        xg = self.gcn(xg, sup)
        x  = xg.reshape(B,S,N,d).permute(0,3,2,1)
        x  = self.bn(x)
        return self.res_conv(x) + res, self.skip_conv(x)


# ─── STFormer ────────────────────────────────────────────────────────────
# Follows paper exactly:
#   Spatial:  for each timestep s, multi-head attention across N nodes
#             + adjacency-weighted spatial position embedding
#   Temporal: for each node n, multi-head attention across S timesteps
#             + learned temporal position embedding (positional encoding)

class SpatialTransformerLayer(nn.Module):
    """
    Paper eq 15-19.
    Input x: (B, d, N, S)
    Spatial position embedding: x_s = x + A @ W_spe  (A broadcasts over batch/time)
    Then: x_s → MHA → add&norm → FFN → add&norm
    """
    def __init__(self, d, n_heads, n_nodes, dropout=0.1):
        super().__init__()
        self.W_spe  = nn.Linear(n_nodes, d, bias=False)   # A(N,N) @ W(N,d)→(N,d)
        self.attn   = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff     = nn.Sequential(nn.Linear(d,d*2), nn.GELU(),
                                    nn.Dropout(dropout), nn.Linear(d*2,d))
        self.norm1  = nn.LayerNorm(d)
        self.norm2  = nn.LayerNorm(d)

    def forward(self, x, A_buf):       # x:(B,d,N,S), A_buf:(N,N)
        B, d, N, S = x.shape
        # Spatial position embedding: inject graph structure
        # A_buf: (N,N) → W_spe: (N,d) → broadcast → (1,d,N,1)
        spe = self.W_spe(A_buf.T).T.unsqueeze(0).unsqueeze(-1)   # (1,d,N,1)
        x   = x + spe
        # Flatten to (B*S, N, d) for attention across N
        xt  = x.permute(0,3,2,1).reshape(B*S, N, d)
        a, _ = self.attn(xt, xt, xt)
        xt  = self.norm1(xt + a)
        xt  = self.norm2(xt + self.ff(xt))
        return xt.reshape(B,S,N,d).permute(0,3,2,1)   # (B,d,N,S)


class TemporalTransformerLayer(nn.Module):
    """
    Paper eq 20-25.
    Input x: (B, d, N, S)
    Temporal position embedding: learned (S, d) added to each node's sequence
    Then: x_t → MHA → add&norm → FFN → add&norm
    """
    def __init__(self, d, n_heads, seq_len, dropout=0.1):
        super().__init__()
        self.pos_emb = nn.Embedding(seq_len, d)           # learned position
        self.attn    = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff      = nn.Sequential(nn.Linear(d,d*2), nn.GELU(),
                                     nn.Dropout(dropout), nn.Linear(d*2,d))
        self.norm1   = nn.LayerNorm(d)
        self.norm2   = nn.LayerNorm(d)
        self.register_buffer('pos_idx', torch.arange(seq_len))

    def forward(self, x):              # x:(B,d,N,S)
        B, d, N, S = x.shape
        pe  = self.pos_emb(self.pos_idx[:S])              # (S, d)
        pe  = pe.T.unsqueeze(0).unsqueeze(2)              # (1,d,1,S)
        x   = x + pe
        # Flatten to (B*N, S, d) for attention across S
        xt  = x.permute(0,2,3,1).reshape(B*N, S, d)
        a, _ = self.attn(xt, xt, xt)
        xt  = self.norm1(xt + a)
        xt  = self.norm2(xt + self.ff(xt))
        return xt.reshape(B,N,S,d).permute(0,3,1,2)       # (B,d,N,S)


class STFormer(nn.Module):
    """
    L alternating spatial + temporal transformer layers (paper Section IV-C).
    Each layer: spatial → temporal, with residual through the stack.
    """
    def __init__(self, d, n_heads, n_nodes, seq_len, n_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleList([
                SpatialTransformerLayer(d, n_heads, n_nodes, dropout),
                TemporalTransformerLayer(d, n_heads, seq_len, dropout)
            ])
            for _ in range(n_layers)
        ])

    def forward(self, x, A_buf):       # x:(B,d,N,S)
        for sp, tp in self.layers:
            x = tp(sp(x, A_buf))
        return x


print('DiffusionGCN, WaveBlock, SpatialTransformer, TemporalTransformer, STFormer defined.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GWNet v25 — Architecture
#
# Input: x_rec, x_hour, x_day  (B, S, N, F)  +  tod, dow  (B, S)
#
# Flow:
#  1. 3 × start_conv  → additive fusion  → + tod/dow emb + node_emb → x
#  2. 16 WaveBlocks (dilated TCN + DiffGCN)  →  residual x + skip_i list
#  3. sum(skip_i) → skip_proj → STFormer (L=2 spatial+temporal layers)
#  4. pool last 4 timesteps → output MLP (2 conv) → (B, 12, 170)
#
# Key improvements over v20 (which reached MAE=13.6):
#  + 3-stream multi-period input (captures daily/hourly periodicity)
#  + STFormer: global spatial+temporal attention after WaveBlocks
#  + Fixed metrics (pool all predictions before computing MAE)
#  + Huber loss for smoother gradients near convergence
#  + ReduceLROnPlateau only (no cosine fighting)
# ═══════════════════════════════════════════════════════════════════════════

class GWNetV25(nn.Module):
    def __init__(self, cfg, A_dist_np):
        super().__init__()
        N, d = cfg.num_nodes, cfg.d_model

        # ── Static adjacency ──────────────────────────────────────
        A   = torch.FloatTensor(A_dist_np)
        Df  = A.sum(1, keepdim=True).clamp(min=1e-8)
        Db  = A.T.sum(1, keepdim=True).clamp(min=1e-8)
        self.register_buffer('A_fwd', A   / Df)
        self.register_buffer('A_bwd', A.T / Db)
        # Raw normalised A used for spatial position embedding in STFormer
        self.register_buffer('A_spe', A   / Df)

        # ── Adaptive adjacency ────────────────────────────────────
        self.E1 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)
        self.E2 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)

        # ── 3-stream input projections ────────────────────────────
        self.sc_rec  = nn.Conv2d(cfg.in_features, d, (1,1))
        self.sc_hour = nn.Conv2d(cfg.in_features, d, (1,1))
        self.sc_day  = nn.Conv2d(cfg.in_features, d, (1,1))
        self.node_emb = nn.Parameter(torch.randn(1, d, N, 1) * 0.01)

        # ── Temporal embeddings (tod + dow → d_model) ─────────────
        self.tod_emb   = nn.Embedding(288, cfg.d_time)
        self.dow_emb   = nn.Embedding(7,   cfg.d_time)
        self.time_proj = nn.Linear(cfg.d_time * 2, d)

        # ── WaveBlocks (v20 proven: 16 blocks, dilations×4) ───────
        dilations = ([1,2,4,8] * 4)[:cfg.n_layers]
        self.blocks = nn.ModuleList([
            WaveBlock(d, cfg.d_skip, cfg.kernel_size, dil,
                      cfg.n_supports, cfg.gcn_order, cfg.dropout)
            for dil in dilations
        ])

        # ── Skip → STFormer projection ────────────────────────────
        self.skip_proj = nn.Conv2d(cfg.d_skip, cfg.d_stf, (1,1))

        # ── STFormer (L=2 spatial+temporal layers) ────────────────
        self.stformer = STFormer(
            d=cfg.d_stf,
            n_heads=cfg.stf_heads,
            n_nodes=N,
            seq_len=cfg.seq_len,
            n_layers=cfg.stf_layers,
            dropout=cfg.dropout * 0.5    # lighter dropout in STFormer
        )

        # ── Output MLP ────────────────────────────────────────────
        self.end_conv1 = nn.Conv2d(cfg.d_stf, cfg.d_end,    (1,1))
        self.end_conv2 = nn.Conv2d(cfg.d_end,  cfg.pred_len, (1,1))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
        nn.init.normal_(self.tod_emb.weight, std=0.01)
        nn.init.normal_(self.dow_emb.weight, std=0.01)

    def get_supports(self):
        A_adp = F.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)
        return [self.A_fwd, self.A_bwd, A_adp]

    def forward(self, x_rec, x_hour, x_day, tod, dow):
        # x_*: (B, S, N, F)
        def to4d(x): return x.permute(0,3,2,1)   # (B, F, N, S)

        # ── 3-stream additive fusion ──────────────────────────────
        x = (self.sc_rec (to4d(x_rec )) +
             self.sc_hour(to4d(x_hour)) +
             self.sc_day (to4d(x_day )))          # (B, d, N, S)
        x = x + self.node_emb

        # ── Temporal embedding (tod + dow) ────────────────────────
        te = torch.cat([self.tod_emb(tod),
                        self.dow_emb(dow)], dim=-1)   # (B, S, 2*d_time)
        te = self.time_proj(te).permute(0,2,1).unsqueeze(2)  # (B,d,1,S)
        x  = x + te

        # ── 16 WaveBlocks ─────────────────────────────────────────
        sup   = self.get_supports()
        skips = []
        for blk in self.blocks:
            x, skip = blk(x, sup)
            skips.append(skip)                    # each: (B, d_skip, N, S)

        # ── Sum all skip connections (B, d_skip, N, S) ────────────
        skip_sum = sum(skips)

        # ── Project down to STFormer dim ──────────────────────────
        h = F.relu(self.skip_proj(skip_sum))      # (B, d_stf, N, S)

        # ── STFormer: spatial+temporal attention ──────────────────
        h = self.stformer(h, self.A_spe)          # (B, d_stf, N, S)

        # ── Pool last 4 timesteps → (B, d_stf, N, 1) ─────────────
        h = h[:, :, :, -4:].mean(dim=-1, keepdim=True)

        # ── Output MLP ────────────────────────────────────────────
        out = F.relu(self.end_conv1(h))            # (B, d_end, N, 1)
        out = self.end_conv2(out).squeeze(-1)      # (B, pred_len, N)
        return out


print('GWNetV25 defined.')


In [ ]:
dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)
mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)

set_seed()
model = GWNetV25(cfg, A_dist_np).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GWNetV25 | {n_params:,} params')

# Smoke test
with torch.no_grad():
    B = 2
    def dummy(): return torch.randn(B, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    tod = torch.randint(0, 288, (B, cfg.seq_len)).to(device)
    dow = torch.randint(0, 7,   (B, cfg.seq_len)).to(device)
    out = model(dummy(), dummy(), dummy(), tod, dow)
    ok  = list(out.shape) == [B, cfg.pred_len, cfg.num_nodes]
    print(f'Output: {out.shape}  {chr(10003) if ok else chr(10007)+" CHECK!"}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# METRICS — pool ALL predictions before computing, avoids batch-mean bias
# ═══════════════════════════════════════════════════════════════════════════

def huber_loss(pred, true, delta=1.0):
    """Huber loss on normalised space. Smoother gradient than MAE near optimum."""
    err  = torch.abs(pred - true)
    return torch.where(err < delta, 0.5*err**2, delta*(err - 0.5*delta)).mean()

def compute_mae(P, Y, thresh=0.0):
    """P, Y: denormalised tensors of ANY shape."""
    mask = (Y.abs() > thresh).float()
    return (torch.abs(P-Y)*mask).sum() / (mask.sum()+1e-8)

def compute_rmse(P, Y, thresh=0.0):
    mask = (Y.abs() > thresh).float()
    return (((P-Y)**2*mask).sum() / (mask.sum()+1e-8)).sqrt()

def compute_mape(P, Y, thresh=10.0):
    mask = (Y.abs() > thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0)
    return (torch.abs((P-Y)/(Y.abs()+1.0))*mask).sum() / mask.sum() * 100

print('Metrics (pooled-prediction style) defined.')


In [ ]:
scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device,  non_blocking=True)
        x_hour = x_hour.to(device, non_blocking=True)
        x_day  = x_day.to(device,  non_blocking=True)
        y      = y.to(device,      non_blocking=True)
        tod    = tod.to(device,    non_blocking=True)
        dow    = dow.to(device,    non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
            loss = huber_loss(pred, y, delta=cfg.huber_delta)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device, mean_flow, std_flow):
    """
    CRITICAL FIX: pool ALL batch predictions before computing metrics.
    Previous versions computed mean(batch_MAEs) which biases results.
    """
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device,  non_blocking=True)
        x_hour = x_hour.to(device, non_blocking=True)
        x_day  = x_day.to(device,  non_blocking=True)
        tod    = tod.to(device,    non_blocking=True)
        dow    = dow.to(device,    non_blocking=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        # Denormalise on CPU to save GPU VRAM
        pd_ = pred.float().cpu() * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        yd_ = y.float()          * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        all_pred.append(pd_)
        all_true.append(yd_)
    # Pool all predictions → single metric computation
    P = torch.cat(all_pred, dim=0)   # (total_samples, 12, 170)
    Y = torch.cat(all_true, dim=0)
    mae  = compute_mae(P, Y).item()
    rmse = compute_rmse(P, Y).item()
    mape = compute_mape(P, Y).item()
    return mae, rmse, mape

print('Train/eval (pooled metrics) defined.')


In [ ]:
set_seed()

optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# ── LR schedule: warmup 5ep → ReduceLROnPlateau ──────────────────────────
# Proven approach: warmup avoids early instability, plateau only fires when
# val MAE genuinely stops improving. No cosine restarts fighting plateau.
def lr_lambda(ep):
    return min(1.0, (ep+1) / cfg.warmup_eps)

warmup_sched  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=20, factor=0.7, min_lr=1e-5, verbose=False)

best_val_mae = best_val_rmse = best_val_mape = float('inf')
patience_cnt = 0
history      = {'train_loss':[], 'val_mae':[], 'val_rmse':[], 'val_mape':[]}

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GWNet v25 | {params:,} params')
print(f'd={cfg.d_model} d_skip={cfg.d_skip} d_stf={cfg.d_stf} | n_layers={cfg.n_layers}')
print(f'3-stream | STFormer L={cfg.stf_layers} h={cfg.stf_heads} | Huber delta={cfg.huber_delta}')
print(f'AdamW lr={cfg.lr} | warmup={cfg.warmup_eps}ep | plateau(p=20,f=0.7) | wd={cfg.weight_decay}')
print(f'epochs={cfg.epochs} patience={cfg.patience}')
print(f'Baseline -> MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*70)

for epoch in range(1, cfg.epochs+1):
    tr_loss = train_epoch(model, dl_train, optimizer, device)
    val_mae, val_rmse, val_mape = eval_epoch(model, dl_val, device, mean_flow, std_flow)

    if epoch <= cfg.warmup_eps:
        warmup_sched.step()
    else:
        plateau_sched.step(val_mae)

    history['train_loss'].append(tr_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae  = val_mae
        best_val_rmse = val_rmse
        best_val_mape = val_mape
        patience_cnt  = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  <- best'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stop at epoch {epoch}.')
            break

    lr_now = optimizer.param_groups[0]['lr']
    beat   = ' *** BEAT BASELINE! ***' if val_mae < 13.114 else ''
    if epoch % 5 == 0 or tag:
        print(f'Ep {epoch:03d} | Loss={tr_loss:.4f} | '
              f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% '
              f'lr={lr_now:.1e}{tag}{beat}')

print(f'\nBest Val: MAE={best_val_mae:.3f} RMSE={best_val_rmse:.3f} MAPE={best_val_mape:.2f}%')
print(f'Baseline: MAE=13.114   RMSE=22.623   MAPE=8.471%')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].plot(history['train_loss'], 'steelblue', label='Train Loss (Huber)')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['val_mae'], 'tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(history['val_rmse'], 'tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.tight_layout(); plt.savefig('v25_curves.png', dpi=150); plt.show()


In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_eval(model, loader, device, mean_flow, std_flow):
    """Paper-style: pool all predictions, then compute averaged metrics."""
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);  x_hour = x_hour.to(device)
        x_day  = x_day.to(device);  tod    = tod.to(device); dow = dow.to(device)
        pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_  = pred.float().cpu() * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        yd_  = y.float()          * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        all_pred.append(pd_); all_true.append(yd_)
    P = torch.cat(all_pred); Y = torch.cat(all_true)
    mae  = compute_mae(P, Y).item()
    rmse = compute_rmse(P, Y).item()
    mape = compute_mape(P, Y).item()
    print('\n' + '='*55)
    print('  TEST RESULTS  — averaged over all 12 horizons')
    print('='*55)
    print(f'  MAE  : {mae:.3f}    baseline: 13.114   Delta={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline: 22.623   Delta={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline:  8.471%  Delta={mape-8.471:+.3f}%')
    print('='*55)
    return mae, rmse, mape

paper_eval(model, dl_test, device, mean_flow, std_flow)


In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);  x_hour = x_hour.to(device)
        x_day  = x_day.to(device);  tod    = tod.to(device); dow = dow.to(device)
        pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_  = pred.float().cpu() * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        yd_  = y.float()          * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        all_pred.append(pd_); all_true.append(yd_)
    P = torch.cat(all_pred); Y = torch.cat(all_true)
    print(f'{'Horizon':>14} | {'MAE':>8} | {'RMSE':>8} | {'MAPE':>9}')
    print('-'*50)
    for h, lbl in zip([2,5,11], ['3-step(15min)','6-step(30min)','12-step(60min)']):
        mae  = compute_mae(P[:,h,:], Y[:,h,:]).item()
        rmse = compute_rmse(P[:,h,:],Y[:,h,:]).item()
        mape = compute_mape(P[:,h,:],Y[:,h,:]).item()
        print(f'{lbl:>14} | {mae:>8.3f} | {rmse:>8.3f} | {mape:>8.2f}%')

horizon_eval(model, dl_test, device, mean_flow, std_flow)
